# MaskGen Token Regret Critic (Single-Notebook Workflow)

This notebook is a self-contained implementation for:
- loading pretrained MaskGen-VQ + TA-TiTok + CLIP
- training a token-level regret critic while freezing MaskGen
- optional critic-guided remasking refinement at inference
- generating official GenEval-format inputs and running official evaluation

The base generator remains frozen; only the critic head is optimized.

In [ ]:
import os
import json
import math
import random
import subprocess
import sys
from collections import defaultdict
import glob

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import open_clip
import io
import re

from datasets import load_dataset, load_dataset_builder # Added import for Hugging Face Datasets


try:
    import webdataset as wds
except Exception:
    wds = None

from modeling.tatitok import TATiTok
from modeling.maskgen import MaskGen_VQ, get_masking_ratio, open_clip_text_encoding

HF_CACHE_DIR = os.path.abspath(os.environ.get('HF_CACHE_DIR', 'hf_cache'))
LOCAL_FILES_ONLY = bool(HF_CACHE_DIR)
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = HF_CACHE_DIR
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE_DIR
os.environ['OPENCLIP_CACHE_DIR'] = HF_CACHE_DIR
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('HF cache:', HF_CACHE_DIR)
print('LOCAL_FILES_ONLY:', LOCAL_FILES_ONLY)

In [ ]:
def set_global_seed(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_pretrained_stack(root_dir=HF_CACHE_DIR):
    tok_dir = os.path.join(root_dir, 'turkeyju/tokenizer_tatitok_bl128_vq')
    gen_dir = os.path.join(root_dir, 'turkeyju/generator_maskgen_vq_xl')

    print('Loading TA-TiTok tokenizer...')
    tatitok_vq_tokenizer = TATiTok.from_pretrained(pretrained_model_name_or_path=tok_dir, cache_dir=tok_dir)
    tatitok_vq_tokenizer.eval()
    tatitok_vq_tokenizer.requires_grad_(False)

    print('Loading MaskGen-VQ generator...')
    maskgen_vq_generator = MaskGen_VQ.from_pretrained(pretrained_model_name_or_path=gen_dir, cache_dir=gen_dir)
    maskgen_vq_generator.eval()
    maskgen_vq_generator.requires_grad_(False)

    print('Loading CLIP text encoder...')
    clip_encoder, _, _ = open_clip.create_model_and_transforms(
        'ViT-L-14-336', pretrained='openai', force_quick_gelu=True
    )
    del clip_encoder.visual
    clip_tokenizer = open_clip.get_tokenizer('ViT-L-14-336')
    clip_encoder.transformer.batch_first = False
    clip_encoder.eval()
    clip_encoder.requires_grad_(False)

    print('Pretrained stack ready.')
    return tatitok_vq_tokenizer, maskgen_vq_generator, clip_tokenizer, clip_encoder


tatitok_vq_tokenizer, maskgen_vq_generator, clip_tokenizer, clip_encoder = load_pretrained_stack()
maskgen_vq_generator = maskgen_vq_generator.to(device)
tatitok_vq_tokenizer = tatitok_vq_tokenizer.to(device)
clip_encoder = clip_encoder.to(device)
print('Generator device:', next(maskgen_vq_generator.parameters()).device)
print('Tokenizer device:', next(tatitok_vq_tokenizer.parameters()).device)
print('CLIP text encoder device:', next(clip_encoder.parameters()).device)

In [ ]:
class TokenRegretCritic(nn.Module):
    def __init__(self, hidden_dim, text_dim, timestep_dim=32, mlp_dim=512, logits_topk=8, use_hidden=True):
        super().__init__()
        self.use_hidden = bool(use_hidden)
        self.logits_topk = int(logits_topk)
        self.timestep_dim = int(timestep_dim)
        in_dim = self.timestep_dim + self.logits_topk + 2 + text_dim
        if self.use_hidden:
            in_dim += hidden_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, mlp_dim),
            nn.SiLU(),
            nn.Linear(mlp_dim, mlp_dim),
            nn.SiLU(),
            nn.Linear(mlp_dim, 1),
        )

    def _timestep_embedding(self, timesteps):
        half = self.timestep_dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=timesteps.device, dtype=torch.float32) / max(half, 1))
        args = timesteps.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.timestep_dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb

    def _logit_features(self, logits):
        k = min(self.logits_topk, logits.shape[-1])
        topk = torch.topk(logits, k=k, dim=-1).values
        if k < self.logits_topk:
            topk = F.pad(topk, (0, self.logits_topk - k))
        probs = torch.softmax(logits, dim=-1)
        entropy = -(probs * torch.log(probs.clamp(min=1e-8))).sum(dim=-1, keepdim=True)
        max_prob = probs.max(dim=-1, keepdim=True).values
        return torch.cat([topk, entropy, max_prob], dim=-1)

    def forward(self, hidden_states, logits, timesteps, text_features):
        bsz, seq_len, _ = logits.shape
        t_feat = self._timestep_embedding(timesteps).unsqueeze(1).expand(bsz, seq_len, -1)
        logit_feat = self._logit_features(logits)
        text_feat = text_features.unsqueeze(1).expand(bsz, seq_len, -1)
        chunks = [t_feat, logit_feat, text_feat]
        if self.use_hidden:
            if hidden_states is None:
                raise ValueError('use_hidden=True but hidden_states is None')
            chunks.insert(1, hidden_states)
        x = torch.cat(chunks, dim=-1)
        return self.net(x).squeeze(-1)

def build_token_regret_critic(model, use_hidden=True, timestep_dim=32, mlp_dim=512, logits_topk=8):
    hidden_dim = int(model.pos_embed.shape[-1])
    text_dim = int(model.pos_embed.shape[-1])
    return TokenRegretCritic(
        hidden_dim=hidden_dim,
        text_dim=text_dim,
        timestep_dim=timestep_dim,
        mlp_dim=mlp_dim,
        logits_topk=logits_topk,
        use_hidden=use_hidden,)
    

In [ ]:
def prepare_text_guidance(text, clip_tokenizer, clip_encoder, device):
    '''Encode text using CLIP text encoder and return the resulting features.
        Based on Offical MaskGen code:'''
    text_tokens = clip_tokenizer(text).to(device)
    cast_dtype = clip_encoder.transformer.get_cast_dtype()
    text_embed = clip_encoder.token_embedding(text_tokens).to(cast_dtype)
    text_embed = text_embed + clip_encoder.positional_embedding.to(cast_dtype)
    text_embed = text_embed.permute(1, 0, 2)
    text_embed = clip_encoder.transformer(text_embed, attn_mask=clip_encoder.attn_mask)
    text_embed = text_embed.permute(1, 0, 2)
    text_guidance = clip_encoder.ln_final(text_embed)
    return text_guidance.to(device)


def _resolve_num_selected(seq_len, ratio_or_count, min_tokens=1):
    value = float(ratio_or_count)
    if value <= 0:
        return 0
    if value <= 1:
        count = int(seq_len * value)
    else:
        count = int(value)
    count = max(int(min_tokens), count)
    return min(int(seq_len), count)


def extract_token_nll(logits, target_tokens):
    return F.cross_entropy(logits.transpose(1, 2), target_tokens, reduction='none')


def select_token_subset(candidate_mask, sample_ratio, min_tokens=1):
    if candidate_mask.dtype != torch.bool:
        candidate_mask = candidate_mask.bool()
    bsz, seq_len = candidate_mask.shape
    k = _resolve_num_selected(seq_len, sample_ratio, min_tokens=min_tokens)
    if k == 0:
        return torch.zeros(bsz, 0, dtype=torch.long, device=candidate_mask.device), torch.zeros(bsz, 0, dtype=torch.bool, device=candidate_mask.device)
    scores = torch.rand(bsz, seq_len, device=candidate_mask.device)
    scores = scores.masked_fill(~candidate_mask, -1.0)
    idx = scores.topk(k=k, dim=-1).indices
    valid = candidate_mask.gather(dim=-1, index=idx)
    return idx, valid


def select_topk_token_positions(scores, remask_ratio, candidate_mask=None, min_tokens=1):
    bsz, seq_len = scores.shape
    k = _resolve_num_selected(seq_len, remask_ratio, min_tokens=min_tokens)
    if k == 0:
        return torch.zeros(bsz, 0, dtype=torch.long, device=scores.device), torch.zeros(bsz, 0, dtype=torch.bool, device=scores.device)
    if candidate_mask is None:
        candidate_mask = torch.ones_like(scores, dtype=torch.bool)
    elif candidate_mask.dtype != torch.bool:
        candidate_mask = candidate_mask.bool()
    masked_scores = scores.masked_fill(~candidate_mask, float('-inf'))
    idx = masked_scores.topk(k=k, dim=-1).indices
    valid = candidate_mask.gather(dim=-1, index=idx)
    return idx, valid


def remask_positions(tokens, indices, mask_token_id, valid_mask=None):
    out = tokens.clone()
    if valid_mask is None:
        valid_mask = torch.ones_like(indices, dtype=torch.bool)
    gathered = out.gather(dim=-1, index=indices)
    masked = torch.full_like(gathered, int(mask_token_id))
    update = torch.where(valid_mask, masked, gathered)
    return out.scatter(dim=-1, index=indices, src=update)


    
@torch.no_grad()
def forward_features_vq(model, input_ids, condition, condition_pooled, aesthetic_score=None):
    embeddings = model.embeddings(input_ids)
    cond = model.text_embed_proj(condition)
    pooled = condition_pooled
    if model.micro_condition:
        pooled = model.concat_micro_cond(pooled, aesthetic_score)
    pooled = model.cond_pooled_proj(pooled)

    x = embeddings + model.pos_embed[:, :embeddings.shape[1]]
    for blk in model.blocks:
        cond, x = blk(x, cond, pooled.squeeze(1))
    x = model.norm(x, pooled.squeeze(1))
    logits = model.lm_head(x)
    return logits, x, pooled.squeeze(1)


@torch.no_grad()
def forward_with_cfg_features(model, ids, condition, condition_pooled, none_cond, none_cond_pooled, cfg_scale, timestep_value, sample_aesthetic_score):
    num_samples = ids.shape[0]
    aes = None
    if sample_aesthetic_score is not None and model.micro_condition:
        aes_val = float(sample_aesthetic_score)
        if cfg_scale != 0:
            aes = torch.full((num_samples * 2,), aes_val, device=ids.device)
        else:
            aes = torch.full((num_samples,), aes_val, device=ids.device)

    if cfg_scale != 0:
        logits_all, hidden_all, text_all = forward_features_vq(model, torch.cat([ids, ids], dim=0), torch.cat([condition, none_cond], dim=0), torch.cat([condition_pooled, none_cond_pooled], dim=0), aesthetic_score=aes,)
        cond_logits, uncond_logits = logits_all[:num_samples], logits_all[num_samples:]
        logits = cond_logits + (cond_logits - uncond_logits) * cfg_scale
        hidden = hidden_all[:num_samples]
        text_feat = text_all[:num_samples]
    else:
        logits, hidden, text_feat = forward_features_vq(model, ids, condition, condition_pooled, aesthetic_score=aes, )
    timesteps = torch.full((num_samples,), float(timestep_value), device=ids.device)
    return logits, hidden, text_feat, timesteps


def _add_gumbel_noise(logits, temperature):
    eps = 1e-20
    noise = torch.rand_like(logits)
    g = -torch.log(-torch.log(noise.clamp(min=eps)).clamp(min=eps))
    return logits + float(temperature) * g


def refine_tokens_with_critic(
    model,
    draft_ids,
    captions,
    clip_tokenizer,
    clip_encoder,
    critic,
    remask_ratio=0.05,
    refine_loops=1,
    refine_start_step=10,
    num_sample_steps=16,
    guidance_scale=12.0,
    sample_aesthetic_score=6.5,
    randomize_temperature=1.5,
    critic_use_hidden=True,
    score_gate_quantile=0.8,
    score_gate_min=None,
    refine_softmax_temperature=0.7,
    use_gumbel_in_refine=False,
):
    condition, condition_pooled = open_clip_text_encoding(clip_tokenizer, clip_encoder, captions)
    none_cond, none_cond_pooled = open_clip_text_encoding(clip_tokenizer, clip_encoder, [''])
    none_cond = none_cond.repeat(condition.shape[0], 1, 1)
    none_cond_pooled = none_cond_pooled.repeat(condition.shape[0], 1, 1)

    ids = draft_ids.clone()
    for loop_idx in range(int(refine_loops)):
        # Keep refinement in a late-stage regime instead of replaying early noisy steps.
        ratio = float(refine_start_step + loop_idx + 1) / float(max(num_sample_steps, 1))
        ratio = max(0.75, min(0.95, ratio))

        logits, hidden, text_feat, timesteps = forward_with_cfg_features(
            model, ids, condition, condition_pooled, none_cond, none_cond_pooled,
            cfg_scale=float(guidance_scale),
            timestep_value=ratio,
            sample_aesthetic_score=sample_aesthetic_score,
        )

        scores = critic(
            hidden_states=hidden if critic_use_hidden else None,
            logits=logits,
            timesteps=timesteps,
            text_features=text_feat,
        )

        # Higher regret scores mean "more useful to remask". Do not flip the sign.
        mask_token_id = int(model.mask_token_id)
        candidate_mask = ids.ne(mask_token_id)
        select_idx, select_valid = select_topk_token_positions(
            scores,
            remask_ratio,
            candidate_mask=candidate_mask,
        )
        select_scores = scores.gather(dim=-1, index=select_idx)

        if score_gate_quantile is not None:
            gate_threshold = torch.quantile(
                scores,
                q=max(0.0, min(1.0, float(score_gate_quantile))),
                dim=-1,
                keepdim=True,
            )
            select_valid = select_valid & (select_scores >= gate_threshold)

        if score_gate_min is not None:
            select_valid = select_valid & (select_scores >= float(score_gate_min))

        ids = remask_positions(ids, select_idx, model.mask_token_id, valid_mask=select_valid)

        logits_resample, _, _, _ = forward_with_cfg_features(
            model,
            ids,
            condition,
            condition_pooled,
            none_cond,
            none_cond_pooled,
            cfg_scale=float(guidance_scale),
            timestep_value=ratio,
            sample_aesthetic_score=sample_aesthetic_score,
        )

        # Never let refinement re-sample the mask token itself if it is part of the decode vocabulary.
        mask_token_id = int(model.mask_token_id)
        if 0 <= mask_token_id < logits_resample.shape[-1]:
            logits_resample[..., mask_token_id] = -1e9
        logits_resample = logits_resample / float(refine_softmax_temperature)

        if use_gumbel_in_refine:
            sampled_ids = _add_gumbel_noise(logits_resample, float(randomize_temperature)).argmax(dim=-1)
        else:
            sampled_ids = logits_resample.argmax(dim=-1)

        is_mask = ids.eq(model.mask_token_id)
        ids = torch.where(is_mask, sampled_ids, ids)

    if ids.eq(int(model.mask_token_id)).any():
        raise RuntimeError('Refinement left mask tokens in the final token grid.')

    return ids


@torch.no_grad()
def generate_image_vq_batch(prompts, model, tokenizer, clip_tokenizer, clip_encoder, guidance_scale=12.0, randomize_temperature=1.5, aesthetic_score=6.5, num_sample_steps=16, use_regret_remask=False, critic=None, remask_ratio=0.05, refine_loops=1, refine_start_step=10, critic_use_hidden=True, score_gate_quantile=0.8, score_gate_min=None, refine_softmax_temperature=0.7, use_gumbel_in_refine=False, device='cuda'):
    model_device = next(model.parameters()).device
    if next(clip_encoder.parameters()).device != model_device:
        clip_encoder = clip_encoder.to(model_device)
    if next(tokenizer.parameters()).device != model_device:
        tokenizer = tokenizer.to(model_device)
    if isinstance(device, str):
        device = model_device
    tokens = model.generate(captions=prompts, guidance_scale=guidance_scale, randomize_temperature=randomize_temperature, sample_aesthetic_score=aesthetic_score, num_sample_steps=num_sample_steps, clip_tokenizer=clip_tokenizer, clip_encoder=clip_encoder)
    tokens = tokens.to(model_device)
    if use_regret_remask:
        if critic is None:
            raise ValueError('use_regret_remask=True but critic is None')
        critic_device = next(critic.parameters()).device
        if critic_device != model_device:
            critic = critic.to(model_device)
        tokens = refine_tokens_with_critic(
            model=model,
            draft_ids=tokens,
            captions=prompts,
            clip_tokenizer=clip_tokenizer,
            clip_encoder=clip_encoder,
            critic=critic,
            remask_ratio=remask_ratio,
            refine_loops=refine_loops,
            refine_start_step=refine_start_step,
            num_sample_steps=int(num_sample_steps),
            guidance_scale=float(guidance_scale),
            sample_aesthetic_score=float(aesthetic_score),
            randomize_temperature=float(randomize_temperature),
            critic_use_hidden=bool(critic_use_hidden),
            score_gate_quantile=score_gate_quantile,
            score_gate_min=score_gate_min,
            refine_softmax_temperature=refine_softmax_temperature,
            use_gumbel_in_refine=use_gumbel_in_refine,
        )
    text_guidance = prepare_text_guidance(prompts, clip_tokenizer, clip_encoder, model_device)
    image = tokenizer.decode_tokens(tokens, text_guidance)
    image = torch.clamp(image, 0.0, 1.0)
    image = (image * 255.0).permute(0, 2, 3, 1).to('cpu', dtype=torch.uint8).numpy()
    return [Image.fromarray(arr) for arr in image]

In [ ]:

def sample_masked_state(model, gt_tokens, timesteps):
    bsz, seq_len = gt_tokens.shape
    mask_ratio = get_masking_ratio(timesteps, model.mask_schedule_strategy)
    mask_ratio = torch.clamp(mask_ratio, min=1e-6, max=1.0)
    num_masked = (seq_len * mask_ratio).round().clamp(min=1)
    randperm = torch.rand(bsz, seq_len, device=gt_tokens.device).argsort(dim=-1)
    masks = randperm < num_masked.unsqueeze(1)
    z_t = torch.where(masks, torch.full_like(gt_tokens, int(model.mask_token_id)), gt_tokens)
    return z_t, masks


def pairwise_rank_loss(scores, regrets, valid_mask, margin=0.05):
    total = scores.new_tensor(0.0)
    count = 0
    for b in range(scores.shape[0]):
        keep = valid_mask[b]
        if keep.sum() < 2:
            continue
        s = scores[b][keep]
        r = regrets[b][keep]
        higher = (r.unsqueeze(1) - r.unsqueeze(0)) > 0
        if higher.any():
            loss_mat = F.relu(float(margin) - (s.unsqueeze(1) - s.unsqueeze(0)))
            total = total + loss_mat[higher].mean()
            count += 1
    if count == 0:
        return scores.new_tensor(0.0)
    return total / count


def save_critic_checkpoint(path, critic, optimizer, step, config_dict):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    payload = {
        'critic': critic.state_dict(),
        'optimizer': optimizer.state_dict(),
        'step': int(step),
        'config': config_dict,
    }
    torch.save(payload, path)


def load_critic_checkpoint(path, critic, optimizer=None, map_location='cpu'):
    ckpt = torch.load(path, map_location=map_location)
    critic.load_state_dict(ckpt['critic'], strict=True)
    if optimizer is not None and 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    return ckpt

def load_trained_critic(ckpt_path, model, use_hidden=True):
    target_device = next(model.parameters()).device
    critic = build_token_regret_critic(model, use_hidden=use_hidden).to(target_device)
    _ = load_critic_checkpoint(ckpt_path, critic, optimizer=None, map_location=target_device)
    critic.eval()
    critic.requires_grad_(False)
    return critic


def _default_hf_shard_urls(num_shards=69):
    base_url = (
        'https://huggingface.co/datasets/ProGamerGov/'
        'synthetic-dataset-1m-high-quality-captions/'
        'resolve/main/data/data-{i:06d}.tar'
    )
    return [base_url.format(i=i) for i in range(int(num_shards))]


def _infer_total_examples_from_urls(urls):
    if urls is None or len(urls) == 0:
        return None
    try:
        builder = load_dataset_builder('webdataset', data_files={'train': list(urls)})
        split_info = None
        if getattr(builder, 'info', None) is not None and getattr(builder.info, 'splits', None) is not None:
            split_info = builder.info.splits.get('train')
        if split_info is not None and getattr(split_info, 'num_examples', None) is not None:
            num_examples = int(split_info.num_examples)
            if num_examples > 0:
                return num_examples
    except Exception:
        pass

    # Known fallback for the official 1M shard list when metadata is unavailable.
    try:
        default_urls = _default_hf_shard_urls()
        if len(urls) == len(default_urls) and all(str(a) == str(b) for a, b in zip(urls, default_urls)):
            return 1_000_000
    except Exception:
        pass
    return None


def _extract_caption_from_sample(sample):
    for key in ('txt', 'caption', 'text', 'prompt'):
        val = sample.get(key)
        if isinstance(val, str) and val.strip() != '':
            return val.strip()

    meta = sample.get('json')
    if isinstance(meta, str):
        try:
            meta = json.loads(meta)
        except Exception:
            meta = {}
    if isinstance(meta, dict):
        for key in ('short_caption', 'long_caption', 'caption', 'text', 'prompt'):
            val = meta.get(key)
            if isinstance(val, str) and val.strip() != '':
                return val.strip()
    return None


def _extract_pil_image_from_sample(sample):
    for key in ('jpg', 'jpeg', 'png', 'webp', 'image'):
        if key not in sample:
            continue
        obj = sample[key]
        if isinstance(obj, Image.Image):
            return obj.convert('RGB')
        if isinstance(obj, dict) and 'bytes' in obj and obj['bytes'] is not None:
            return Image.open(io.BytesIO(obj['bytes'])).convert('RGB')
    return None


def _center_crop_arr(pil_image, image_size):
    while min(*pil_image.size) >= 2 * image_size:
        pil_image = pil_image.resize(tuple(x // 2 for x in pil_image.size), resample=Image.BOX)
    scale = image_size / min(*pil_image.size)
    pil_image = pil_image.resize(tuple(round(x * scale) for x in pil_image.size), resample=Image.BICUBIC)
    arr = np.array(pil_image)
    crop_y = (arr.shape[0] - image_size) // 2
    crop_x = (arr.shape[1] - image_size) // 2
    return Image.fromarray(arr[crop_y: crop_y + image_size, crop_x: crop_x + image_size])


def _tokenizer_image_size(tokenizer):
    cfg = getattr(tokenizer, 'config', None)
    size = None
    if cfg is not None:
        try:
            size = int(cfg.model.vq_model.image_size)
        except Exception:
            size = None
    return size if size is not None and size > 0 else 256


def _images_to_tokens(tokenizer, images, expected_seq_len):
    image_size = _tokenizer_image_size(tokenizer)
    tokenizer_device = next(tokenizer.parameters()).device
    proc = []
    for img in images:
        crop = _center_crop_arr(img.convert('RGB'), image_size)
        arr = np.asarray(crop, dtype=np.float32) / 255.0
        ten = torch.from_numpy(arr).permute(2, 0, 1)
        proc.append(ten)
    x = torch.stack(proc, dim=0).to(tokenizer_device, non_blocking=True)

    with torch.no_grad():
        _, result_dict = tokenizer.encode(x)
    token_grid = result_dict['min_encoding_indices'].long()
    tokens = token_grid.view(token_grid.shape[0], -1)

    if tokens.shape[1] != int(expected_seq_len):
        raise ValueError(
            f'Token length mismatch: got {tokens.shape[1]}, expected {int(expected_seq_len)}. '
            'Check tokenizer/image size compatibility with the generator.'
        )
    return tokens


def _iter_hf_stream_batches(urls, batch_size, max_examples=None, repeat=False, rank=0, world_size=1):
    batch_images = []
    batch_captions = []
    seen = 0
    seen_all = 0
    limit = None if max_examples is None else int(max_examples)

    def _yield_or_buffer(sample):
        nonlocal seen, seen_all, batch_images, batch_captions
        if limit is not None and seen >= limit:
            return False
        image = _extract_pil_image_from_sample(sample)
        caption = _extract_caption_from_sample(sample)
        if image is None or caption is None:
            return True
        if int(world_size) > 1:
            if (seen_all % int(world_size)) != int(rank):
                seen_all += 1
                return True
        seen_all += 1
        batch_images.append(image)
        batch_captions.append(caption)
        seen += 1
        if len(batch_images) >= int(batch_size):
            out = (batch_images, batch_captions)
            batch_images, batch_captions = [], []
            return out
        return True

    if urls is None or len(urls) == 0:
        urls = _default_hf_shard_urls(num_shards=69)

    pass_idx = 0
    while True:
        pass_idx += 1
        stream_source = 'webdataset'

        try:
            if wds is not None:
                dataset = wds.WebDataset(urls, shardshuffle=False).decode('pil')
            else:
                stream_source = 'datasets streaming'
                dataset = load_dataset('webdataset', data_files={'train': urls}, split='train', streaming=True)

            for sample in dataset:
                res = _yield_or_buffer(sample)
                if res is False:
                    break
                if isinstance(res, tuple):
                    yield res
        except Exception as e:
            if wds is not None:
                stream_source = 'datasets streaming fallback'
                err_name = type(e).__name__
                print('webdataset stream failed, falling back to datasets streaming:', err_name)
                dataset = load_dataset('webdataset', data_files={'train': urls}, split='train', streaming=True)
                for sample in dataset:
                    res = _yield_or_buffer(sample)
                    if res is False:
                        break
                    if isinstance(res, tuple):
                        yield res
            else:
                raise RuntimeError(
                    'datasets streaming failed and webdataset is unavailable. ' + str(e)
                )

        if limit is not None and seen >= limit:
            break
        if not bool(repeat):
            break
        print(f'Restarting HF stream pass #{pass_idx + 1} after exhaustion ({stream_source})')

    if len(batch_images) > 0:
        yield batch_images, batch_captions


def train_token_regret_critic_head(
    model,
    clip_tokenizer,
    clip_encoder,
    train_shards_path_or_url,
    eval_shards_path_or_url='coco',
    output_dir='outputs/token_regret_critic',
    resume_checkpoint=None,
    num_epochs=1,
    per_gpu_batch_size=16,
    learning_rate=2e-4,
    token_sample_ratio=0.2,
    lambda_rank=0.1,
    rank_margin=0.05,
    counterfactual_chunk_size=512,
    save_every=250,
    seed=42,
    critic_use_hidden=True,
    enable_dp=None,
):
    set_global_seed(seed)

    class _MaskGenFeatureExtractor(nn.Module):
        def __init__(self, base_model):
            super().__init__()
            self.base_model = base_model

        def forward(self, input_ids, condition, condition_pooled, aesthetic_score=None):
            return forward_features_vq(self.base_model, input_ids, condition, condition_pooled, aesthetic_score)

    for p in model.parameters():
        p.requires_grad = False
    model.eval()

    if torch.cuda.is_available() and next(model.parameters()).device.type != 'cuda':
        model = model.to(device)
    if torch.cuda.is_available() and next(tatitok_vq_tokenizer.parameters()).device.type != 'cuda':
        tatitok_vq_tokenizer.to(device)
    if torch.cuda.is_available() and next(clip_encoder.parameters()).device.type != 'cuda':
        clip_encoder.to(device)

    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
    dist_enabled = dist.is_available() and dist.is_initialized()
    rank = dist.get_rank() if dist_enabled else 0
    world_size = dist.get_world_size() if dist_enabled else 1
    local_rank = int(os.environ.get('LOCAL_RANK', rank if gpu_count > 0 else 0))

    if torch.cuda.is_available():
        local_rank = max(0, min(local_rank, gpu_count - 1))
        model_device = torch.device(f'cuda:{local_rank}')
    else:
        model_device = torch.device('cpu')

    model = model.to(model_device)
    tatitok_vq_tokenizer.to(model_device)
    clip_encoder.to(model_device)

    if enable_dp is None:
        enable_dp = (gpu_count > 1)
    use_ddp = bool(enable_dp and gpu_count > 1 and dist_enabled and model_device.type == 'cuda')
    is_main_process = (rank == 0)

    if is_main_process:
        print('Training model device:', model_device)
        print('Tokenizer device:', next(tatitok_vq_tokenizer.parameters()).device)
        print('CLIP encoder device:', next(clip_encoder.parameters()).device)

    if bool(enable_dp) and gpu_count > 1 and not dist_enabled and is_main_process:
        print('Multi-GPU requested but torch.distributed is not initialized. Launch with torchrun to use DDP; falling back to single-GPU mode.')

    feature_model = _MaskGenFeatureExtractor(model).to(model_device)
    critic = build_token_regret_critic(model, use_hidden=critic_use_hidden).to(model_device)

    if use_ddp:
        critic = DDP(critic, device_ids=[local_rank], output_device=local_rank, broadcast_buffers=False, find_unused_parameters=False)
        if is_main_process:
            print(f'DDP enabled with world_size={world_size}, local_rank={local_rank}')
    elif is_main_process:
        print('Single-process training mode')

    critic.train()

    optimizer = torch.optim.AdamW(critic.parameters(), lr=float(learning_rate), weight_decay=1e-4)
    start_step = 0
    if resume_checkpoint is not None and str(resume_checkpoint).strip() != '':
        critic_for_load = critic.module if isinstance(critic, DDP) else critic
        ckpt = load_critic_checkpoint(resume_checkpoint, critic_for_load, optimizer=optimizer, map_location=model_device)
        start_step = int(ckpt.get('step', 0))
        if is_main_process:
            print('Resumed critic from step', start_step)

    if train_shards_path_or_url is None:
        urls = _default_hf_shard_urls()
    elif isinstance(train_shards_path_or_url, (list, tuple)):
        urls = [str(x).strip() for x in train_shards_path_or_url if str(x).strip()]
    else:
        raise ValueError('train_shards_path_or_url must be a list/tuple of full shard URLs.')
    if len(urls) == 0:
        raise ValueError('No training shard URL resolved from train_shards_path_or_url.')

    requested_epochs = max(1, int(num_epochs))
    if int(per_gpu_batch_size) <= 0:
        raise ValueError('per_gpu_batch_size must be > 0')

    inferred_dataset_examples = _infer_total_examples_from_urls(urls)
    if inferred_dataset_examples is not None and is_main_process:
        print(f'Inferred dataset size: {int(inferred_dataset_examples)} examples from dataset metadata/shards')

    steps_per_epoch_estimate = None
    if inferred_dataset_examples is not None:
        effective_world_size = int(world_size) if use_ddp else 1
        steps_per_epoch_estimate = max(1, math.ceil(int(inferred_dataset_examples) / (int(per_gpu_batch_size) * effective_world_size)))
        if is_main_process:
            print(f'Estimated iterations per epoch: {int(steps_per_epoch_estimate)} at batch size {int(per_gpu_batch_size)} per rank')

    os.makedirs(output_dir, exist_ok=True)
    cfg = {
        'train_shards_path_or_url': train_shards_path_or_url,
        'resolved_train_urls_count': len(urls),
        'resolved_train_urls_head': urls[:3],
        'eval_shards_path_or_url': eval_shards_path_or_url,
        'num_epochs': int(requested_epochs),
        'dataset_size_inferred': None if inferred_dataset_examples is None else int(inferred_dataset_examples),
        'steps_per_epoch_estimate': None if steps_per_epoch_estimate is None else int(steps_per_epoch_estimate),
        'per_gpu_batch_size': int(per_gpu_batch_size),
        'learning_rate': float(learning_rate),
        'token_sample_ratio': float(token_sample_ratio),
        'lambda_rank': float(lambda_rank),
        'rank_margin': float(rank_margin),
        'counterfactual_chunk_size': int(counterfactual_chunk_size),
        'critic_use_hidden': bool(critic_use_hidden),
        'gpu_count': int(gpu_count),
        'distributed_data_parallel': bool(use_ddp),
        'world_size': int(world_size),
        'rank': int(rank),
        'data_pipeline': 'webdataset.primary+datasets.streaming.fallback->on_the_fly_tokens',
    }
    if is_main_process:
        with open(os.path.join(output_dir, 'train_config.json'), 'w') as f:
            json.dump(cfg, f, indent=2)

    step = int(start_step)
    total_steps_estimate = None if steps_per_epoch_estimate is None else int(steps_per_epoch_estimate) * int(requested_epochs)
    if is_main_process:
        if total_steps_estimate is not None:
            print(f'Total iterations: {int(total_steps_estimate)}')
        else:
            print('Total iterations: unknown (dataset size metadata unavailable)')
    pbar = tqdm(total=total_steps_estimate, initial=step, desc='iterations', unit='iter', disable=not is_main_process)

    default_aes = float(getattr(model, 'sample_aesthetic_score', 6.5))

    for epoch_idx in range(int(requested_epochs)):
        stream_iter = _iter_hf_stream_batches(
            urls=urls,
            batch_size=int(per_gpu_batch_size),
            max_examples=None,
            repeat=False,
            rank=(rank if use_ddp else 0),
            world_size=(world_size if use_ddp else 1),
        )
        epoch_steps = 0
        for images, captions in stream_iter:

            gt_tokens = _images_to_tokens(
                tokenizer=tatitok_vq_tokenizer,
                images=images,
                expected_seq_len=int(model.image_seq_len),
            ).to(model_device, non_blocking=True)

            aes_scores = None
            if model.micro_condition:
                aes_scores = torch.full((gt_tokens.shape[0],), default_aes, device=model_device)

            with torch.no_grad():
                condition, condition_pooled = model.preprocess_condition(captions, clip_tokenizer, clip_encoder)
                condition = condition.to(model_device, non_blocking=True)
                condition_pooled = condition_pooled.to(model_device, non_blocking=True)
                timesteps = torch.rand((gt_tokens.shape[0],), device=model_device)
                z_t, _ = sample_masked_state(model, gt_tokens, timesteps)
                logits_orig, hidden_orig, text_feat = feature_model(z_t, condition, condition_pooled, aes_scores)

            candidate_mask = z_t.ne(int(model.mask_token_id))
            selected_idx, selected_valid = select_token_subset(candidate_mask, sample_ratio=float(token_sample_ratio), min_tokens=1)

            with torch.no_grad():
                nll_orig = extract_token_nll(logits_orig, gt_tokens)
                nll_orig_selected = nll_orig.gather(dim=-1, index=selected_idx)
                regrets = torch.zeros_like(nll_orig_selected)
                pair = torch.nonzero(selected_valid, as_tuple=False)
                for start in range(0, pair.shape[0], int(counterfactual_chunk_size)):
                    part = pair[start:start + int(counterfactual_chunk_size)]
                    if part.numel() == 0:
                        continue
                    b_idx = part[:, 0]
                    k_idx = part[:, 1]
                    tok_idx = selected_idx[b_idx, k_idx]

                    z_cf = z_t[b_idx].clone()
                    z_cf[torch.arange(z_cf.shape[0], device=z_cf.device), tok_idx] = int(model.mask_token_id)
                    cond_cf = condition[b_idx]
                    pooled_cf = condition_pooled[b_idx]
                    aes_cf = aes_scores[b_idx] if aes_scores is not None else None

                    logits_cf, _, _ = feature_model(z_cf, cond_cf, pooled_cf, aes_cf)
                    gt_cf = gt_tokens[b_idx]
                    nll_cf_all = extract_token_nll(logits_cf, gt_cf)
                    nll_cf_target = nll_cf_all[torch.arange(nll_cf_all.shape[0], device=z_cf.device), tok_idx]
                    regrets[b_idx, k_idx] = nll_orig_selected[b_idx, k_idx] - nll_cf_target
                targets = torch.tanh(regrets)

            pred = critic(
                hidden_states=hidden_orig if critic_use_hidden else None,
                logits=logits_orig,
                timesteps=timesteps,
                text_features=text_feat,
            )
            pred_sel = pred.gather(dim=-1, index=selected_idx)

            denom = selected_valid.float().sum().clamp(min=1.0)
            loss_reg = (((pred_sel - targets) ** 2) * selected_valid.float()).sum() / denom
            loss_rank = pairwise_rank_loss(pred_sel, regrets, selected_valid, margin=float(rank_margin))
            loss = loss_reg + float(lambda_rank) * loss_rank

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            step += 1
            epoch_steps += 1
            pbar.update(1)
            if is_main_process and step % 10 == 0:
                pbar.set_postfix({'loss': float(loss.item()), 'reg': float(loss_reg.item()), 'rank': float(loss_rank.item())})

            if is_main_process and step % int(save_every) == 0:
                critic_for_save = critic.module if isinstance(critic, DDP) else critic
                save_critic_checkpoint(os.path.join(output_dir, f'critic_step_{step}.pt'), critic_for_save, optimizer, step, cfg)

        if is_main_process and epoch_steps == 0:
            print('HF stream produced no batches in one dataset pass')

    if use_ddp:
        dist.barrier()

    critic_for_save = critic.module if isinstance(critic, DDP) else critic
    if is_main_process:
        save_critic_checkpoint(os.path.join(output_dir, 'critic_last.pt'), critic_for_save, optimizer, step, cfg)
    pbar.close()
    critic = critic.module if isinstance(critic, DDP) else critic
    critic.eval()
    if is_main_process:
        print(f'Training done after {requested_epochs} epoch(s). Last checkpoint:', os.path.join(output_dir, 'critic_last.pt'))
    return critic, os.path.join(output_dir, 'critic_last.pt')

In [ ]:
def run_token_regret_sanity_checks(model):
    seq_len = int(model.image_seq_len)
    sample_tokens = torch.arange(2 * seq_len, device=device).reshape(2, seq_len) % int(model.target_codebook_size)

    sample_indices = torch.tensor([[1, 3], [2, 4]], device=device)
    sample_valid = torch.tensor([[True, False], [True, True]], device=device)
    remasked = remask_positions(sample_tokens, sample_indices, model.mask_token_id, valid_mask=sample_valid)
    assert remasked[0, 1].item() == int(model.mask_token_id)
    assert remasked[0, 3].item() == sample_tokens[0, 3].item()
    assert remasked[1, 2].item() == int(model.mask_token_id)
    assert remasked[1, 4].item() == int(model.mask_token_id)

    critic = build_token_regret_critic(model, use_hidden=True).to(device)
    dummy_scores = critic(
        hidden_states=torch.randn(2, seq_len, model.pos_embed.shape[-1], device=device),
        logits=torch.randn(2, seq_len, model.target_codebook_size, device=device),
        timesteps=torch.rand(2, device=device),
        text_features=torch.randn(2, model.pos_embed.shape[-1], device=device),
    )
    assert dummy_scores.shape == (2, seq_len)
    print('Token regret sanity checks passed.')


run_token_regret_sanity_checks(maskgen_vq_generator)

## Critic Training Entry Cell

Official mode: train only the critic head from the provided Hugging Face WebDataset shards using on-the-fly tokenizer encoding via `datasets.load_dataset(..., streaming=True)`. Bootstrap mode is disabled.

In [ ]:
# Hugging Face source config (official training)
GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 1
ENABLE_DP = (GPU_COUNT > 1)
PER_GPU_BATCH_SIZE = 256
TRAIN_CRITIC_TRAIN_SHARDS = _default_hf_shard_urls()
RESUME_CKPT = 'outputs/token_regret_critic/critic_step_200.pt'

trained_critic = None
trained_critic_ckpt = None
if ENABLE_DP:
    ddp_cmd = [
        sys.executable,
        '-m', 'torch.distributed.run',
        f'--nproc_per_node={GPU_COUNT}',
        'scripts/train_token_regret_ddp.py',
        '--num-epochs', '1',
        '--per-gpu-batch-size', str(PER_GPU_BATCH_SIZE),
        '--learning-rate', '2e-4',
        '--token-sample-ratio', '0.2',
        '--lambda-rank', '0.1',
        '--rank-margin', '0.05',
        '--counterfactual-chunk-size', '128',
        '--save-every', '50',
        '--seed', '42',
        '--output-dir', 'outputs/token_regret_critic',
    ]
    if RESUME_CKPT and os.path.isfile(RESUME_CKPT):
        ddp_cmd += ['--resume-checkpoint', RESUME_CKPT]
    ddp_env = os.environ.copy()
    ddp_env['DIST_BACKEND'] = 'gloo'
    print('Launching DDP training with gloo backend:', ' '.join(ddp_cmd))
    run = subprocess.run(ddp_cmd, check=False, env=ddp_env)
    if run.returncode != 0:
        raise RuntimeError(f'DDP training failed with return code {run.returncode}')
    trained_critic_ckpt = 'outputs/token_regret_critic/critic_last.pt'
else:
    trained_critic, trained_critic_ckpt = train_token_regret_critic_head(
        model=maskgen_vq_generator,
        clip_tokenizer=clip_tokenizer,
        clip_encoder=clip_encoder,
        train_shards_path_or_url=TRAIN_CRITIC_TRAIN_SHARDS,
        eval_shards_path_or_url='coco',
        output_dir='outputs/token_regret_critic',
        resume_checkpoint=RESUME_CKPT if os.path.isfile(RESUME_CKPT) else None,
        num_epochs=1,
        per_gpu_batch_size=PER_GPU_BATCH_SIZE,
        learning_rate=2e-4,
        token_sample_ratio=0.2,
        lambda_rank=0.1,
        rank_margin=0.05,
        counterfactual_chunk_size=128,
        save_every=10,
        seed=42,
        critic_use_hidden=True,
        enable_dp=False,
    )

effective_global_batch = PER_GPU_BATCH_SIZE * (GPU_COUNT if ENABLE_DP else 1)
print('Detected GPUs:', GPU_COUNT)
print('Per-GPU training batch size:', PER_GPU_BATCH_SIZE)
print('Effective global batch size:', effective_global_batch)
print('DDP enabled:', ENABLE_DP)
if ENABLE_DP:
    print('DDP backend: gloo')
print('Starting official critic training from HF shards:', TRAIN_CRITIC_TRAIN_SHARDS)
if trained_critic_ckpt is not None:
    print('Trained critic checkpoint:', trained_critic_ckpt)

In [ ]:
# Quick train-shard comparison: without head (baseline) vs with head (critic-guided)
import matplotlib.pyplot as plt


NUM_SAMPLES = 8
NUM_SAMPLE_STEPS = 16
GUIDANCE_SCALE = 12.0
RANDOMIZE_TEMPERATURE = 2.0
AESTHETIC_SCORE = 6.5
COMPARE_SEED = 42
CRITIC_CKPT_CANDIDATES = [
    globals().get('trained_critic_ckpt', None),
    'outputs/token_regret_critic/critic_step_200.pt',
    'outputs/token_regret_critic_smoke/critic_last.pt',
]
REMASK_RATIO = 0.2
REFINE_LOOPS = 1
REFINE_START_STEP = 8

DEFAULT_TRAIN_SHARDS = _default_hf_shard_urls()

train_shards = globals().get('TRAIN_CRITIC_TRAIN_SHARDS', DEFAULT_TRAIN_SHARDS)
if not isinstance(train_shards, (list, tuple)):
    raise ValueError('TRAIN_CRITIC_TRAIN_SHARDS must be a list/tuple of full shard URLs.')
urls = [str(x).strip() for x in train_shards if str(x).strip()]
if len(urls) == 0:
    raise RuntimeError('No train shard URLs resolved.')

stream = _iter_hf_stream_batches(urls=urls, batch_size=NUM_SAMPLES, max_examples=NUM_SAMPLES * 4)
_, sample_captions = next(stream)
prompts = [str(x).strip() for x in sample_captions if str(x).strip()][:NUM_SAMPLES]
if len(prompts) == 0:
    raise RuntimeError('Could not extract captions from train shards.')

print(f'Generating {len(prompts)} baseline samples (without head)...')
set_global_seed(COMPARE_SEED)
images_no_head = generate_image_vq_batch(
    prompts=prompts,
    model=maskgen_vq_generator,
    tokenizer=tatitok_vq_tokenizer,
    clip_tokenizer=clip_tokenizer,
    clip_encoder=clip_encoder,
    guidance_scale=GUIDANCE_SCALE,
    randomize_temperature=RANDOMIZE_TEMPERATURE,
    aesthetic_score=AESTHETIC_SCORE,
    num_sample_steps=NUM_SAMPLE_STEPS,
    use_regret_remask=False,
    critic=None,
    device=device,
 )

CRITIC_CKPT_PATH = None
for _p in CRITIC_CKPT_CANDIDATES:
    if _p is None:
        continue
    _p = str(_p).strip()
    if _p and os.path.isfile(_p):
        CRITIC_CKPT_PATH = _p
        break
if CRITIC_CKPT_PATH is None:
    raise FileNotFoundError('No critic checkpoint found. Checked: ' + ', '.join(str(x) for x in CRITIC_CKPT_CANDIDATES))

print(f'Loading critic head from: {CRITIC_CKPT_PATH}')
critic_head = load_trained_critic(CRITIC_CKPT_PATH, maskgen_vq_generator, use_hidden=True)
model_device = next(maskgen_vq_generator.parameters()).device
critic_head = critic_head.to(model_device)

print(f'Generating {len(prompts)} critic-guided samples (with head)...')
set_global_seed(COMPARE_SEED)
images_with_head = generate_image_vq_batch(
    prompts=prompts,
    model=maskgen_vq_generator,
    tokenizer=tatitok_vq_tokenizer,
    clip_tokenizer=clip_tokenizer,
    clip_encoder=clip_encoder,
    guidance_scale=GUIDANCE_SCALE,
    randomize_temperature=RANDOMIZE_TEMPERATURE,
    aesthetic_score=AESTHETIC_SCORE,
    num_sample_steps=NUM_SAMPLE_STEPS,
    use_regret_remask=True,
    critic=critic_head,
    remask_ratio=REMASK_RATIO,
    refine_loops=REFINE_LOOPS,
    refine_start_step=REFINE_START_STEP,
    score_gate_quantile=None,
    score_gate_min=None,
    critic_use_hidden=True,
    device=device,
 )

n = len(prompts)
fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
axes = np.array(axes).reshape(n, 2)

for i in range(n):
    # Left: baseline (no critic head)
    ax_left = axes[i, 0]
    ax_left.imshow(images_no_head[i])
    ax_left.axis('off')
    cap = prompts[i].replace('\n', ' ').strip()
    if len(cap) > 100:
        cap = cap[:97] + '...'
    ax_left.set_title('Without Head | ' + cap, fontsize=9)

    # Right: critic-guided (with head)
    ax_right = axes[i, 1]
    ax_right.imshow(images_with_head[i])
    ax_right.axis('off')
    ax_right.set_title('With Head', fontsize=9)

plt.tight_layout()
plt.show()

# Define Official GenEval Evalute Method

In [ ]:
def run_official_geneval_eval(eval_dir, out_file, eval_python='/data/conda_envs/miniconda3/envs/geneval_official/bin/python', cuda_visible_devices=None):
    eval_script = os.path.join('geneval-official', 'evaluation', 'evaluate_images.py')
    if not os.path.exists(eval_script):
        raise FileNotFoundError('Missing evaluator script: ' + eval_script)
    if not os.path.exists(eval_python):
        raise FileNotFoundError('Missing python: ' + eval_python)

    if not os.path.isdir(eval_dir):
        raise FileNotFoundError(f'Evaluation input directory not found: {eval_dir}')

    model_path = os.path.join(os.getcwd(), 'geneval-official', 'models')
    mmdet_root = os.path.join('geneval-official', 'mmdetection-2x')

    # Try multiple known config locations in priority order.
    candidates = [
        os.path.join(mmdet_root, 'configs', 'mask2former', 'mask2former_swin-s-p4-w7-224_lsj_8x2_50e_coco.py'),
        os.path.join(os.getcwd(), 'geneval-official', 'mmdetection-2x', 'configs', 'mask2former', 'mask2former_swin-s-p4-w7-224_lsj_8x2_50e_coco.py'),
    ]

    # Probe config path inside the target eval python env (mmdet installs configs under .mim).
    probe_code = (
        "import mmdet, pathlib; "
        "root = pathlib.Path(mmdet.__file__).resolve().parent; "
        "p = root / '.mim' / 'configs' / 'mask2former' / 'mask2former_swin-s-p4-w7-224_lsj_8x2_50e_coco.py'; "
        "print(str(p) if p.exists() else '')"
    )
    probe = subprocess.run([eval_python, '-c', probe_code], capture_output=True, text=True)
    probed_cfg = probe.stdout.strip()
    if probed_cfg:
        candidates.append(probed_cfg)

    model_config = None
    for c in candidates:
        if c and os.path.isfile(c):
            model_config = c
            break

    print(f'Evaluation directory: {eval_dir}')
    print(f'Model path: {model_path}')
    print(f'MMDetection root: {mmdet_root} (exists: {os.path.isdir(mmdet_root)})')
    print('Resolved model config:', model_config if model_config else '<not found>')

    if model_config is None:
        raise FileNotFoundError(
            'Could not find Mask2Former config. Expected one of: '\
            + '\n- ' + '\n- '.join(str(x) for x in candidates if x)
        )

    env = os.environ.copy()
    if os.path.isdir(mmdet_root):
        env['PYTHONPATH'] = mmdet_root + (':' + env['PYTHONPATH'] if 'PYTHONPATH' in env and env['PYTHONPATH'] else '')
    if cuda_visible_devices is not None:
        env['CUDA_VISIBLE_DEVICES'] = str(cuda_visible_devices)

    preflight = subprocess.run(
        [eval_python, '-c', "import mmcv, mmdet; print('mmcv', mmcv.__version__, 'mmdet', mmdet.__version__)"],
        env=env, capture_output=True, text=True
    )
    if preflight.returncode != 0:
        print('Preflight check FAILED:')
        print(preflight.stdout)
        print(preflight.stderr)
        raise RuntimeError('Preflight failed for mmcv/mmdet environment. Is the geneval_official conda environment activated?')
    print('Preflight check passed:', preflight.stdout.strip())

    cmd = [
        eval_python,
        eval_script,
        eval_dir,
        '--outfile', out_file,
        '--model-path', model_path,
        '--model-config', model_config,
        '--options',
        'threshold=0.3',
        'counting_threshold=0.9',
        'max_objects=16',
        'max_overlap=1.0',
        'position_threshold=0.1',
    ]

    os.makedirs(os.path.dirname(out_file), exist_ok=True)

    print('Running official evaluator...')
    print('Command:', ' '.join(cmd))
    run = subprocess.run(cmd, env=env, capture_output=True, text=True)

    stderr_file = out_file.replace('.jsonl', '_eval_stderr.log')
    with open(stderr_file, 'w') as f:
        f.write(run.stderr)

    if run.returncode != 0:
        print('Evaluator FAILED with return code:', run.returncode)
        print('\nStdout (first 2000 chars):')
        print(run.stdout[:2000])
        print('\nStderr (first 2000 chars):')
        print(run.stderr[:2000])
        print(f'\nFull stderr saved to: {stderr_file}')
        raise RuntimeError(f'Official evaluation failed (return code {run.returncode}). Check stderr at {stderr_file}')
    print(run.stdout)

    rows = []
    with open(out_file, 'r') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    if len(rows) == 0:
        raise RuntimeError('No rows found in result file: ' + out_file)

    tag_prompt_any = defaultdict(dict)
    prompt_sample_count = defaultdict(int)
    for r in rows:
        tag = r['tag']
        meta = r['metadata']
        corr = bool(r['correct'])
        if meta not in tag_prompt_any[tag]:
            tag_prompt_any[tag][meta] = False
        tag_prompt_any[tag][meta] = tag_prompt_any[tag][meta] or corr
        prompt_sample_count[meta] += 1

    expected = 4
    bad = {k: v for k, v in prompt_sample_count.items() if v != expected}
    if bad:
        print('Warning: prompts not matching expected 4 samples. First mismatches:', list(bad.items())[:10])

    tag_scores = {}
    for tag, mp in tag_prompt_any.items():
        vals = list(mp.values())
        tag_scores[tag] = sum(vals) / max(len(vals), 1)

    overall = sum(tag_scores.values()) / max(len(tag_scores), 1)
    print('Prompt-level any aggregation scores:')
    for tag in sorted(tag_scores.keys()):
        print(f'  {tag}: {tag_scores[tag]:.4f}')
    print('Overall:', round(overall, 5))

    summary = {
        'outfile': out_file,
        'overall_prompt_level': overall,
        'tag_scores': tag_scores,
        'num_rows': len(rows),
        'num_prompts': len(prompt_sample_count),
    }
    summary_path = os.path.splitext(out_file)[0] + '_paper_protocol_summary.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print('Summary JSON:', summary_path)
    return summary